# EXP-013 — SVM cân bằng trọng số lớp trên WESAD

**Người thực hiện:** Nam  
**Nhóm:** 03 — Class Balancing  
**Trạng thái:** Chưa chạy trong bản mã nguồn phát hành.

## 1. Mục tiêu và giả thuyết

Kiểm tra việc dùng `class_weight="balanced"` trong SVM RBF có cải thiện Macro-F1 và Balanced Accuracy so với EXP-004 (SVM mặc định) trên cùng WESAD physiological68 hay không.

## 2. Thiết kế thí nghiệm

- Bộ G2: 68 đặc trưng, 15 người, 6.189 cửa sổ theo bộ được nhóm nghiệm thu.
- Outer LOSO: huấn luyện 14 người, kiểm tra người còn lại (15 folds × 2 tasks = 30).
- `SimpleImputer` và `StandardScaler` được fit trên train mỗi fold.
- SVM RBF mặc định `C=1.0`, `gamma="scale"`; chỉ đổi `class_weight="balanced"`.
- Không tuning, không oversampling. Nhãn nhị phân 0/1 và ba lớp 1/2/3.

## 3. Khởi tạo môi trường và đọc cấu hình

In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'scripts/run_experiment.py').is_file() and (p/'src/wesad/models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy wesad-stress-detection. Hãy mở notebook trong repo đúng.')


def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def execute(args):
    command=[sys.executable,*args]
    print('Lệnh:', ' '.join(map(str,command)))
    proc=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print('Return code:',proc.returncode)
    if proc.stdout: print('STDOUT: \n'+proc.stdout[-12000:])
    if proc.stderr: print('STDERR: \n'+proc.stderr[-12000:])
    if proc.returncode: raise RuntimeError(f'Chạy thất bại ({proc.returncode}). Xem STDERR ở trên.')
    return proc

CONFIG_PATH=ROOT/'experiments/03_balancing/EXP-013_SVM/config.yaml'
EXPORT_DIR=CONFIG_PATH.parent
MAIN_DIR=ROOT/'experiments/03_balancing/EXP-013_SVM/run_outputs/main'
assert CONFIG_PATH.is_file()
print(CONFIG_PATH)

c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\03_balancing\EXP-013_SVM\config.yaml


## 4. Kiểm tra dữ liệu và checksum

In [2]:
CONFIG=yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
DATA=ROOT/CONFIG['features']; SCHEMA=ROOT/CONFIG['schema']
for f in [DATA,SCHEMA]:
    if not f.is_file(): raise FileNotFoundError(f'Thiếu dữ liệu G2: {f}')
actual=sha256(DATA)
assert actual==CONFIG['feature_sha256'],f'Không khớp checksum: {actual}'
sys.path.insert(0,str(ROOT/'src'))
from wesad.data import load_feature_table
frame, features=load_feature_table(DATA,SCHEMA)
assert len(features)==68 and frame.subject_id.nunique()==15
print('Số cửa sổ:',len(frame),'| Số người:',frame.subject_id.nunique(),'| Số đặc trưng:',len(features))
display(frame.label_original.value_counts().sort_index().rename('Số cửa sổ').to_frame())


Số cửa sổ: 6189 | Số người: 15 | Số đặc trưng: 68


,Số cửa sổ
label_original,
1,3370
2,1848
3,971


## 5. Kiểm tra cấu hình SVM

In [3]:
from wesad.models import build_pipeline
assert CONFIG['models']==['svm'] and CONFIG['balance'] is True and CONFIG['nested'] is False
balanced=build_pipeline('svm',CONFIG)
normal_config=dict(CONFIG);normal_config['balance']=False
baseline=build_pipeline('svm',normal_config)
print('EXP-004 baseline:',baseline['model'])
print('EXP-013 class-balanced:',balanced['model'])
assert balanced['model'].get_params()['class_weight']=='balanced'


EXP-004 baseline: SVC()
EXP-013 class-balanced: SVC(class_weight='balanced')


## 6. R01 — Chạy thử một fold S2

Bật `RUN_QUICK=True` nếu muốn chạy thử. Output quick nằm ngoài thư mục báo cáo chính, không được xem là 15-fold chính thức.

In [4]:
RUN_QUICK=True
if RUN_QUICK:
    # Quick dùng --outer-subjects và tách output để không lẫn main.
    quick_config=dict(CONFIG)
    quick_config['output_dir']='outputs/notebook_research/EXP-013/quick_s2'
    quick_path=EXPORT_DIR/'quick_s2_config.yaml'
    quick_path.write_text(yaml.safe_dump(quick_config,sort_keys=False,allow_unicode=True),encoding='utf-8')
    execute(['scripts/run_experiment.py','--config',str(quick_path),'--outer-subjects','S2'])
else: print('R01 chưa chạy: bật RUN_QUICK=True nếu cần.')


Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/run_experiment.py --config c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\03_balancing\EXP-013_SVM\quick_s2_config.yaml --outer-subjects S2
Return code: 0
STDOUT: 
multiclass/svm/S2: F1=0.4364
binary/svm/S2: F1=0.5543
Run status: partial



## 7. R02 — Chạy chính thức và xuất CSV/PNG

Chạy 30 LOSO fold. Sau khi runner hoàn tất, tự tạo các file `results.csv`, `summary.csv`, `predictions.csv`, `result_provenance.json`, ảnh PNG **ngay trong thư mục EXP-013_SVM**, giống EXP-007.

In [5]:
RUN_MAIN=True
if RUN_MAIN:
    execute(['scripts/run_experiment.py','--config',str(CONFIG_PATH),'--resume'])
    execute(['scripts/report_exp013.py'])
else: print('Chưa chạy chính thức. Đặt RUN_MAIN=True.')


Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/run_experiment.py --config c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\03_balancing\EXP-013_SVM\config.yaml --resume
Return code: 0
STDOUT: 
multiclass/svm/S10: F1=0.5614
multiclass/svm/S11: F1=0.7110
multiclass/svm/S13: F1=0.5034
multiclass/svm/S14: F1=0.6406
multiclass/svm/S15: F1=0.6743
multiclass/svm/S16: F1=0.3761
multiclass/svm/S17: F1=0.5891
multiclass/svm/S2: F1=0.4364
multiclass/svm/S3: F1=0.7776
multiclass/svm/S4: F1=0.5785
multiclass/svm/S5: F1=0.7768
multiclass/svm/S6: F1=0.9536
multiclass/svm/S7: F1=0.6187
multiclass/svm/S8: F1=0.6165
multiclass/svm/S9: F1=0.3580
binary/svm/S10: F1=1.0000
binary/svm/S11: F1=0.8946
binary/svm/S13: F1=0.6465
binary/svm/S14: F1=0.9884
binary/svm/S15: F1=0.7728
binary/svm/S16: F1=0.8361
binary/svm/S17: F1=0.9333
binary/svm/S2: F1=0.5543
binary/svm/S3: F1=0.9603
binary/svm/S4: F1=0.9729
binary/svm/S5: F1=0.9121
binary/svm/S6: F1=1.0000
binary/s

## 8. Đọc và kiểm chứng kết quả

Nếu đã chạy main trước đó nhưng thiếu CSV/PNG, bật `RUN_REPORT=True` để xuất báo cáo mà không huấn luyện lại.

In [6]:
RUN_REPORT=True
if RUN_REPORT:
    execute(['scripts/report_exp013.py'])
summary_path=EXPORT_DIR/'summary.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
    display(pd.read_csv(EXPORT_DIR/'results.csv').head(10))
else: print('Chưa có summary.csv. Chạy main và export trước.')


Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/report_exp013.py
Return code: 0
STDOUT: 
Bỏ qua so sánh EXP-004 do chưa thể xác minh: EXP-004 baseline không đủ bằng chứng khớp dữ liệu/LOSO
      task model  n_subjects  macro_f1_mean  macro_f1_std  balanced_accuracy_mean  balanced_accuracy_std
multiclass   svm          15       0.611474      0.158767                0.654865               0.152759
    binary   svm          15       0.843374      0.175610                0.866232               0.154492
Đã xuất: C:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\03_balancing\EXP-013_SVM



,task,model,n_subjects,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,multiclass,svm,15,0.611474,0.158767,0.654865,0.152759
1,binary,svm,15,0.843374,0.175610,0.866232,0.154492


,subject_id,task,model,n_test,macro_f1,balanced_accuracy,seconds
0,S10,multiclass,svm,426,0.561359,0.561947,0.3003
1,S11,multiclass,svm,417,0.711016,0.733067,0.2641
2,S13,multiclass,svm,417,0.503437,0.579646,0.2480
3,S14,multiclass,svm,410,0.640626,0.728978,0.2618
4,S15,multiclass,svm,418,0.674313,0.713657,0.2600
5,S16,multiclass,svm,415,0.376094,0.399705,0.2499
6,S17,multiclass,svm,426,0.589086,0.624691,0.2491
7,S2,multiclass,svm,395,0.436440,0.516959,0.2562
8,S3,multiclass,svm,401,0.777645,0.857704,0.2387
9,S4,multiclass,svm,404,0.578473,0.633634,0.2623


## 9. Phân tích biểu đồ và so sánh EXP-004

In [7]:
from IPython.display import Image
figures=sorted((EXPORT_DIR/'figures').glob('*.png'))
print('Số ảnh:',len(figures))
for f in figures: print('-',f.name)
comparison=EXPORT_DIR/'comparison_exp004.csv'
if comparison.is_file():
    display(pd.read_csv(comparison).groupby('task')[['delta_macro_f1','delta_balanced_accuracy']].agg(['mean','std']))
else: print('Chưa tạo comparison_exp004.csv vì thiếu bằng chứng baseline tương thích.')


Số ảnh: 4
- confusion_matrix_binary.png
- confusion_matrix_multiclass.png
- macro_f1_by_subject_binary.png
- macro_f1_by_subject_multiclass.png
Chưa tạo comparison_exp004.csv vì thiếu bằng chứng baseline tương thích.


## 10. Hạn chế và kiểm tra nguồn

Số người chỉ 15; các cửa sổ chồng lấn 55 giây. `class_weight=balanced` chỉ điều chỉnh trọng số lỗi, không sinh thêm cửa sổ. Không kết luận có cải thiện nếu chưa có chênh lệch ghép cặp với EXP-004 trên cùng 15 người, cùng dữ liệu và nhãn.

## 11. Nhận xét và kết luận

### 11.1. Nhận xét kết quả thực nghiệm

Thí nghiệm EXP-013 sử dụng thuật toán Support Vector Machine (SVM) với kernel RBF, áp dụng `class_weight="balanced"` nhằm đánh giá tác động của việc điều chỉnh trọng số lớp đối với khả năng phát hiện stress trên bộ dữ liệu WESAD.

Thí nghiệm sử dụng 68 đặc trưng sinh lý của 15 người tham gia và phương pháp Leave-One-Subject-Out (LOSO). Mỗi lượt đánh giá sử dụng dữ liệu của 14 người để huấn luyện và người còn lại để kiểm tra.

Kết quả đã hoàn thành 30 fold LOSO, bao gồm 15 fold phân loại nhị phân và 15 fold phân loại ba lớp.

| Bài toán | Macro-F1 (Mean ± Std) | Balanced Accuracy (Mean ± Std) |
|---|---|---|
| Nhị phân | **0,8434 ± 0,1756** | **0,8662 ± 0,1545** |
| Ba lớp | **0,6115 ± 0,1588** | **0,6549 ± 0,1528** |

Kết quả cho thấy SVM cân bằng trọng số lớp đạt hiệu suất cao hơn ở bài toán nhị phân so với bài toán ba lớp. Macro-F1 của bài toán nhị phân đạt 0,8434, cao hơn khoảng 0,2319 so với phân loại ba lớp.

Điều này cho thấy việc phân biệt Stress và Non-Stress dễ hơn so với việc nhận diện riêng biệt ba trạng thái Baseline, Stress và Amusement.

Tuy nhiên, độ lệch chuẩn Macro-F1 tương đối lớn ở cả hai bài toán, phản ánh sự khác biệt về hiệu suất giữa các người tham gia. Mô hình vẫn gặp khó khăn trong việc tổng quát hóa đồng đều cho tất cả cá nhân.

### 11.2. So sánh với EXP-004 — SVM Baseline

Để đánh giá hiệu quả của cân bằng trọng số lớp, kết quả EXP-013 được đối chiếu với EXP-004 sử dụng SVM mặc định.

| Bài toán | EXP-004 Macro-F1 | EXP-013 Macro-F1 | Chênh lệch xấp xỉ |
|---|---:|---:|---:|
| Nhị phân | 0,8424 | 0,8434 | +0,0010 |
| Ba lớp | 0,6177 | 0,6115 | −0,0062 |

Kết quả cho thấy việc áp dụng `class_weight="balanced"` không mang lại cải thiện đáng kể về Macro-F1 trung bình trong thiết lập hiện tại.

Đối với bài toán nhị phân, Macro-F1 chỉ tăng khoảng 0,0010. Trong khi đó, ở bài toán ba lớp, Macro-F1 giảm khoảng 0,0062 so với SVM mặc định.

Điều này cho thấy cân bằng trọng số lớp không phải lúc nào cũng cải thiện hiệu suất tổng thể. Việc tăng tầm quan trọng của các lớp ít mẫu có thể thay đổi ranh giới phân loại, nhưng cũng có thể làm giảm khả năng nhận diện chính xác ở một số lớp khác.

Các mức chênh lệch còn nhỏ so với độ biến động giữa người tham gia. Vì vậy, chưa đủ bằng chứng để kết luận EXP-013 tốt hơn hay kém hơn EXP-004 một cách có ý nghĩa thống kê.

### 11.3. Hạn chế

- Bộ dữ liệu WESAD chỉ gồm 15 người tham gia nên khả năng tổng quát hóa ra các quần thể khác còn hạn chế.
- Việc điều chỉnh trọng số lớp có thể làm thay đổi hiệu suất của từng lớp mà không cải thiện Macro-F1 trung bình.
- SVM vẫn sử dụng các siêu tham số mặc định, chưa đánh giá sự kết hợp giữa cân bằng lớp và tối ưu tham số.
- Kết quả có độ lệch chuẩn tương đối cao giữa các người tham gia.

### 11.4. Kết luận

Thí nghiệm EXP-013 đã hoàn thành đánh giá SVM với cơ chế cân bằng trọng số lớp trên 15 người tham gia, gồm 30 fold LOSO.

Mô hình đạt Macro-F1 trung bình **0,8434** ở bài toán nhị phân và **0,6115** ở bài toán ba lớp.

So với EXP-004, phương pháp cân bằng trọng số lớp chỉ mang lại cải thiện rất nhỏ ở bài toán nhị phân và làm giảm nhẹ hiệu suất ở bài toán ba lớp.

**Kết quả cho thấy việc sử dụng `class_weight="balanced"` chưa chứng minh được lợi ích rõ rệt đối với SVM trong thiết lập WESAD hiện tại.** Tuy nhiên, đây vẫn là một kết quả nghiên cứu có giá trị, cho thấy hiệu quả của phương pháp xử lý mất cân bằng lớp phụ thuộc vào đặc điểm dữ liệu và bài toán phân loại.